In [ ]:
import os
import numpy as np
import pandas as pd

try:
    import xarray as xr
except ImportError as exc:
    raise ImportError("xarray is required to read .nc files. Please install it in this environment.") from exc

print("xarray version:", xr.__version__)

In [ ]:
# code to download NOAA GeoE files
import os
import time
import requests
import pandas as pd

NOAA_GEOE_BASE_URL = "https://services.swpc.noaa.gov/netcdf/geoelectric/InterMagEarthScope"
NOAA_GEOE_OUT_DIR = "/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/NOAA_geoE"
os.makedirs(NOAA_GEOE_OUT_DIR, exist_ok=True)

dates_2025 = pd.date_range("2025-01-01", "2025-12-31", freq="D")

downloaded, skipped_existing, missing = [], [], []
for d in dates_2025:
    fname = f"{d.strftime('%Y%m%d')}-empirical-EMTF-2022.12-2022.12.nc"
    url = f"{NOAA_GEOE_BASE_URL}/{fname}"
    out_path = os.path.join(NOAA_GEOE_OUT_DIR, fname)

    if os.path.exists(out_path):
        skipped_existing.append(fname)
        continue

    try:
        with requests.get(url, stream=True, timeout=60) as resp:
            if resp.status_code == 404:
                missing.append(fname)
                continue
            resp.raise_for_status()
            tmp_path = out_path + ".part"
            with open(tmp_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=1024 * 1024):
                    if chunk:
                        f.write(chunk)
            os.replace(tmp_path, out_path)
            downloaded.append(fname)
    except requests.exceptions.RequestException as e:
        print(f"Failed to download {fname}: {e}")
        missing.append(fname)

    time.sleep(0.1)

print("Download summary")
print({
    "downloaded": len(downloaded),
    "already_present": len(skipped_existing),
    "missing_or_failed": len(missing),
})
if missing:
    print("Missing/failed files:", missing)


NOTE: First need to download files from: https://services.swpc.noaa.gov/netcdf/geoelectric/InterMagEarthScope/


In [ ]:

# filename_noaa_geoE = '/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/NOAA_geoE/20240101-empirical-EMTF-2022.12-2022.12.nc'
# filename_noaa_geoE = '/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/NOAA_geoE/20240411-empirical-EMTF-2022.12-2022.12.nc'
filename_noaa_geoE = '/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/NOAA_geoE/20240411-empirical-EMTF-2022.12-2022.12.nc'


In [ ]:
import os
import pandas as pd
import xarray as xr

if not os.path.exists(filename_noaa_geoE):
    raise FileNotFoundError(f"File not found: {filename_noaa_geoE}")

print("Reading:", filename_noaa_geoE)

# Open lazily, then load metadata-driven summary
ds_noaa_geoE = xr.open_dataset(filename_noaa_geoE)

print("\n=== DATASET OVERVIEW ===")
print(ds_noaa_geoE)

print("\n=== GLOBAL ATTRIBUTES ===")
if len(ds_noaa_geoE.attrs) == 0:
    print("No global attributes found.")
else:
    for key, value in ds_noaa_geoE.attrs.items():
        print(f"{key}: {value}")

print("\n=== DIMENSIONS ===")
for dim_name, dim_size in ds_noaa_geoE.sizes.items():
    print(f"{dim_name}: {dim_size}")

print("\n=== COORDINATES ===")
for coord_name, coord_da in ds_noaa_geoE.coords.items():
    print(f"{coord_name}: shape={coord_da.shape}, dtype={coord_da.dtype}")

print("\n=== DATA VARIABLES ===")
for var_name, var_da in ds_noaa_geoE.data_vars.items():
    print(f"\n{var_name}: shape={var_da.shape}, dtype={var_da.dtype}, dims={var_da.dims}")
    if len(var_da.attrs) > 0:
        for k, v in var_da.attrs.items():
            print(f"  - {k}: {v}")

# Build compact variable summary table
var_summary = pd.DataFrame(
    {
        "variable": list(ds_noaa_geoE.data_vars.keys()),
        "dims": [str(ds_noaa_geoE[v].dims) for v in ds_noaa_geoE.data_vars.keys()],
        "shape": [str(tuple(ds_noaa_geoE[v].shape)) for v in ds_noaa_geoE.data_vars.keys()],
        "dtype": [str(ds_noaa_geoE[v].dtype) for v in ds_noaa_geoE.data_vars.keys()],
    }
).sort_values("variable")

print("\n=== VARIABLE SUMMARY TABLE ===")
display(var_summary)

# Preview first variable values (small slice)
first_var = list(ds_noaa_geoE.data_vars.keys())[0] if len(ds_noaa_geoE.data_vars) > 0 else None
if first_var is not None:
    print(f"\n=== SAMPLE SLICE: {first_var} ===")
    da = ds_noaa_geoE[first_var]
    try:
        display(da.isel({d: 0 for d in da.dims if ds_noaa_geoE.sizes[d] > 1}).to_dataframe().head())
    except Exception:
        print(da.values.flat[:10])

# Keep dataset handle available for further exploration
print("\nDataset loaded as variable: ds_noaa_geoE")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd

# Config
time_index = 200
expected_points = 3432

# ---- Helper functions ----
def pick_name(candidates, options):
    for cand in candidates:
        for name in options:
            if cand in name.lower():
                return name
    return None


def reduce_to_point_vector(da, time_name=None, time_index=0, expected_points=3432):
    work = da

    if time_name is not None and time_name in work.dims:
        work = work.isel({time_name: time_index})

    point_dims = [d for d in work.dims if work.sizes[d] == expected_points]
    if len(point_dims) > 0:
        point_dim = point_dims[0]
        for dim in list(work.dims):
            if dim != point_dim:
                work = work.isel({dim: 0})
    else:
        if len(work.dims) > 0:
            point_dim = max(work.dims, key=lambda d: work.sizes[d])
            for dim in list(work.dims):
                if dim != point_dim:
                    work = work.isel({dim: 0})

    work = work.squeeze(drop=True)
    if work.ndim != 1:
        raise ValueError(f"Could not reduce {da.name} to 1D vector. Got shape={work.shape}, dims={work.dims}")

    return np.asarray(work.values, dtype=float)


# ---- Identify fields ----
coord_names = list(ds_noaa_geoE.coords)
var_names = list(ds_noaa_geoE.data_vars)

time_name = pick_name(["time"], coord_names)
lat_name = pick_name(["latitude", "lat"], var_names) or pick_name(["latitude", "lat"], coord_names)
lon_name = pick_name(["longitude", "lon", "long"], var_names) or pick_name(["longitude", "lon", "long"], coord_names)
ex_name = pick_name(["ex", "e_x"], var_names)
ey_name = pick_name(["ey", "e_y"], var_names)

if lat_name is None or lon_name is None or ex_name is None or ey_name is None:
    raise ValueError(
        f"Could not identify required fields. coords={coord_names}, vars={var_names}. "
        f"Detected time={time_name}, lat={lat_name}, lon={lon_name}, Ex={ex_name}, Ey={ey_name}"
    )

# ---- Extract vectors at selected time ----
lat_vec = reduce_to_point_vector(ds_noaa_geoE[lat_name], time_name=time_name, time_index=time_index, expected_points=expected_points)
lon_vec = reduce_to_point_vector(ds_noaa_geoE[lon_name], time_name=time_name, time_index=time_index, expected_points=expected_points)
ex_vec = reduce_to_point_vector(ds_noaa_geoE[ex_name], time_name=time_name, time_index=time_index, expected_points=expected_points)
ey_vec = reduce_to_point_vector(ds_noaa_geoE[ey_name], time_name=time_name, time_index=time_index, expected_points=expected_points)

valid = np.isfinite(lat_vec) & np.isfinite(lon_vec) & np.isfinite(ex_vec) & np.isfinite(ey_vec)
lat_vec = lat_vec[valid]
lon_vec = lon_vec[valid]
ex_vec = ex_vec[valid]
ey_vec = ey_vec[valid]

cmax = 100 #np.nanmax(np.abs(np.r_[ex_vec, ey_vec]))

# ---- Load state boundaries without Cartopy ----
# Prioritize full U.S. Census state shapefile provided by user.
state_boundaries_gdf = None
state_candidates = [
    "/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/location_data/Census_Bureau_Data/tl_2014_us_state/tl_2014_us_state.shp",
    os.path.join("data", "selected_states_May2024_event.geojson"),
    os.path.join("data", "selected_states_MarchApril2023_event.geojson"),
    os.path.join("data", "ca_and_bordering_states.geojson"),
    os.path.join("..", "data", "selected_states_May2024_event.geojson"),
    os.path.join("..", "data", "selected_states_MarchApril2023_event.geojson"),
    os.path.join("..", "data", "ca_and_bordering_states.geojson"),
]

for candidate in state_candidates:
    if os.path.exists(candidate):
        state_boundaries_gdf = gpd.read_file(candidate)
        if state_boundaries_gdf.crs is not None and str(state_boundaries_gdf.crs) != "EPSG:4326":
            state_boundaries_gdf = state_boundaries_gdf.to_crs("EPSG:4326")
        print(f"Using state boundaries file: {candidate}")
        break

# ---- Plot side-by-side maps (no Cartopy required) ----
fig, axes = plt.subplots(2, 1, figsize=(16, 6), constrained_layout=True)

if state_boundaries_gdf is not None:
    for ax in axes:
        state_boundaries_gdf.boundary.plot(ax=ax, color="black", linewidth=0.5, alpha=0.8, zorder=1)

sc0 = axes[0].scatter(
    lon_vec,
    lat_vec,
    c=ex_vec,
    s=14,
    cmap="RdBu_r",
    vmin=-cmax,
    vmax=cmax,
    marker="s",
    linewidths=0,
    zorder=2,
    alpha=0.3
)
axes[0].set_title(f"{ex_name} [mV/km]")
axes[0].set_xlabel("Longitude")
axes[0].set_ylabel("Latitude")
axes[0].grid(True, linestyle="--", alpha=0.3)

sc1 = axes[1].scatter(
    lon_vec,
    lat_vec,
    c=ey_vec,
    s=14,
    cmap="RdBu_r",
    vmin=-cmax,
    vmax=cmax,
    marker="s",
    linewidths=0,
    zorder=2,
    alpha=0.3
)
axes[1].set_title(f"{ey_name} [mV/km]")
axes[1].set_xlabel("Longitude")
axes[1].set_ylabel("Latitude")
axes[1].grid(True, linestyle="--", alpha=0.3)

xmin, xmax = np.nanmin(lon_vec), np.nanmax(lon_vec)
ymin, ymax = np.nanmin(lat_vec), np.nanmax(lat_vec)
for ax in axes:
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

cbar = fig.colorbar(sc1, ax=axes.ravel().tolist(), shrink=0.9)
cbar.set_label("GeoElectric field value")

plt.show()

print(f"Mapped points: {len(lat_vec)}")
print(f"Detected fields -> time: {time_name}, lat: {lat_name}, lon: {lon_name}, Ex: {ex_name}, Ey: {ey_name}")
if time_name is not None:
    time_vals = ds_noaa_geoE[time_name].values
    if len(time_vals) > time_index:
        print(f"Selected time_index={time_index}, time={time_vals[time_index]}")
if state_boundaries_gdf is None:
    print("No state boundary file found. Check the Census shapefile path or add a local GeoJSON under data/.")

In [ ]:
# get GeoE time series (Ex, Ey) at nearest model point to an input lat/lon
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def get_geoe_timeseries_at_latlon(ds, latitude, longitude, expected_points=3432, make_plot=True):
    """
    Return GeoE time series (Ex, Ey) at the nearest grid point to (latitude, longitude).

    Parameters
    ----------
    ds : xarray.Dataset
        Dataset containing lat/lon and GeoE Ex/Ey fields.
    latitude : float
        Target latitude in degrees.
    longitude : float
        Target longitude in degrees.
    expected_points : int
        Typical number of spatial points (used to identify point dimension).
    make_plot : bool
        If True, plot Ex and Ey over time.

    Returns
    -------
    df : pandas.DataFrame
        Columns: time, ex_mV_per_km, ey_mV_per_km, input_lat, input_lon, matched_lat, matched_lon, point_distance_deg
    """

    def pick_name(candidates, options):
        for cand in candidates:
            for name in options:
                if cand in name.lower():
                    return name
        return None

    def to_1d(da):
        work = da

        # Remove time if present in coordinate variables
        if time_name is not None and time_name in work.dims and work.sizes[time_name] > 1:
            work = work.isel({time_name: 0})

        # Reduce to the point dimension
        point_dims = [d for d in work.dims if work.sizes[d] == expected_points]
        if len(point_dims) > 0:
            point_dim = point_dims[0]
        else:
            point_dim = max(work.dims, key=lambda d: work.sizes[d])

        for dim in list(work.dims):
            if dim != point_dim:
                work = work.isel({dim: 0})

        work = work.squeeze(drop=True)
        if work.ndim != 1:
            raise ValueError(f"Could not reduce {da.name} to 1D. shape={work.shape}, dims={work.dims}")

        return np.asarray(work.values, dtype=float), point_dim

    coord_names = list(ds.coords)
    var_names = list(ds.data_vars)

    time_name = pick_name(["time"], coord_names)
    lat_name = pick_name(["latitude", "lat"], var_names) or pick_name(["latitude", "lat"], coord_names)
    lon_name = pick_name(["longitude", "lon", "long"], var_names) or pick_name(["longitude", "lon", "long"], coord_names)
    ex_name = pick_name(["ex", "e_x"], var_names)
    ey_name = pick_name(["ey", "e_y"], var_names)

    if lat_name is None or lon_name is None or ex_name is None or ey_name is None or time_name is None:
        raise ValueError(
            f"Could not identify required fields. coords={coord_names}, vars={var_names}. "
            f"Detected time={time_name}, lat={lat_name}, lon={lon_name}, Ex={ex_name}, Ey={ey_name}"
        )

    # 1) Build point coordinates and find nearest point index
    lat_vec, point_dim_lat = to_1d(ds[lat_name])
    lon_vec, point_dim_lon = to_1d(ds[lon_name])

    if point_dim_lat != point_dim_lon:
        raise ValueError(f"Lat/Lon point dimension mismatch: {point_dim_lat} vs {point_dim_lon}")
    point_dim = point_dim_lat

    valid = np.isfinite(lat_vec) & np.isfinite(lon_vec)
    if not np.any(valid):
        raise ValueError("No valid lat/lon points found in dataset.")

    d2 = (lat_vec[valid] - latitude) ** 2 + (lon_vec[valid] - longitude) ** 2
    valid_indices = np.where(valid)[0]
    nearest_idx = valid_indices[int(np.argmin(d2))]

    matched_lat = float(lat_vec[nearest_idx])
    matched_lon = float(lon_vec[nearest_idx])
    point_distance_deg = float(np.sqrt(np.min(d2)))

    # 2) Extract Ex/Ey time series at that point
    ex_da = ds[ex_name]
    ey_da = ds[ey_name]

    if point_dim not in ex_da.dims or point_dim not in ey_da.dims:
        raise ValueError(
            f"Point dimension '{point_dim}' not present in Ex/Ey dims. Ex dims={ex_da.dims}, Ey dims={ey_da.dims}"
        )
    if time_name not in ex_da.dims or time_name not in ey_da.dims:
        raise ValueError(
            f"Time dimension '{time_name}' not present in Ex/Ey dims. Ex dims={ex_da.dims}, Ey dims={ey_da.dims}"
        )

    ex_series = ex_da.isel({point_dim: nearest_idx})
    ey_series = ey_da.isel({point_dim: nearest_idx})

    # Remove any extra singleton dims
    for dim in list(ex_series.dims):
        if dim != time_name:
            ex_series = ex_series.isel({dim: 0})
    for dim in list(ey_series.dims):
        if dim != time_name:
            ey_series = ey_series.isel({dim: 0})

    time_vals = pd.to_datetime(ds[time_name].values)
    ex_vals = np.asarray(ex_series.values, dtype=float)
    ey_vals = np.asarray(ey_series.values, dtype=float)

    if len(ex_vals) != len(time_vals) or len(ey_vals) != len(time_vals):
        raise ValueError(
            f"Length mismatch: time={len(time_vals)}, Ex={len(ex_vals)}, Ey={len(ey_vals)}"
        )

    df = pd.DataFrame(
        {
            "time": time_vals,
            "ex_mV_per_km": ex_vals,
            "ey_mV_per_km": ey_vals,
            "input_lat": float(latitude),
            "input_lon": float(longitude),
            "matched_lat": matched_lat,
            "matched_lon": matched_lon,
            "point_distance_deg": point_distance_deg,
        }
    )

    if make_plot:
        fig, ax = plt.subplots(figsize=(12, 5))
        ax.plot(df["time"], df["ex_mV_per_km"], label="Ex (mV/km)", linewidth=1.8, color="tab:blue")
        ax.plot(df["time"], df["ey_mV_per_km"], label="Ey (mV/km)", linewidth=1.8, color="tab:orange")
        ax.axhline(0, color="black", linewidth=0.8, alpha=0.7)
        ax.set_title(
            f"GeoE Time Series at input ({latitude:.3f}, {longitude:.3f}) -> nearest ({matched_lat:.3f}, {matched_lon:.3f})"
        )
        ax.set_xlabel("Time")
        ax.set_ylabel("GeoElectric Field (mV/km)")
        ax.grid(True, linestyle="--", alpha=0.3)
        ax.legend()
        plt.tight_layout()
        plt.show()

    return df

# Example usage
sample_lat = 34.05
sample_lon = -118.25
geoe_ts_df = get_geoe_timeseries_at_latlon(ds_noaa_geoE, sample_lat, sample_lon, make_plot=True)
display(geoe_ts_df.head())
print(f"Returned dataframe rows: {len(geoe_ts_df)}")

In [ ]:
# developed in another script, but correctly decodes time, so adding here


# Read one NOAA space-weather file and show decoded timestamps
import re
import numpy as np
import pandas as pd
import xarray as xr

space_weather_file = filename_noaa_geoE #space_files.loc[0].values[1]


def decode_time_index(ds, time_name, fallback_base_date):
    time_da = ds[time_name]

    if np.issubdtype(time_da.dtype, np.datetime64):
        return pd.to_datetime(time_da.values, errors="coerce")

    # Try CF decoding first.
    try:
        decoded = xr.decode_cf(ds[[time_name]])
        vals = decoded[time_name].values
        if np.issubdtype(vals.dtype, np.datetime64):
            return pd.to_datetime(vals, errors="coerce")
    except Exception:
        pass

    vals = np.asarray(time_da.values)

    if np.issubdtype(vals.dtype, np.number):
        vals = vals.astype(float)
        vals[~np.isfinite(vals)] = np.nan
        vals[np.abs(vals) > 1e20] = np.nan

        # Case-insensitive attrs, handles NOAA style keys (UNITS/REFTIME).
        attrs_lc = {str(k).lower(): v for k, v in time_da.attrs.items()}
        units_attr = str(attrs_lc.get("units", "")).lower()
        reftime_attr = attrs_lc.get("reftime", None)

        if "since" in units_attr:
            unit_map = {
                "seconds": "s", "second": "s", "sec": "s",
                "minutes": "m", "minute": "m", "min": "m",
                "hours": "h", "hour": "h", "hr": "h",
                "days": "D", "day": "D",
                "milliseconds": "ms", "millisecond": "ms", "msec": "ms",
                "microseconds": "us", "microsecond": "us", "usec": "us",
                "nanoseconds": "ns", "nanosecond": "ns", "nsec": "ns",
            }
            parsed_unit = None
            for key, val in unit_map.items():
                if key in units_attr:
                    parsed_unit = val
                    break
            if parsed_unit is not None:
                try:
                    base = units_attr.split("since", 1)[1].strip()
                    origin = pd.Timestamp(base)
                    return pd.to_datetime(vals, unit=parsed_unit, origin=origin, errors="coerce")
                except Exception:
                    pass

        # NOAA fallback: seconds offset from REFTIME.
        if reftime_attr is not None:
            try:
                base = pd.Timestamp(str(reftime_attr))
                return base + pd.to_timedelta(vals, unit="s")
            except Exception:
                pass

        # Generic heuristic if metadata is unusable.
        for unit in ("s", "ms", "us", "ns", "h", "m"):
            try:
                candidate = pd.to_datetime(vals, unit=unit, origin="unix", errors="coerce")
                finite = candidate[~pd.isna(candidate)]
                if len(finite) > 0 and finite.min() >= pd.Timestamp("2000-01-01") and finite.max() <= pd.Timestamp("2100-01-01"):
                    return candidate
            except Exception:
                continue

        # Seconds-of-day fallback.
        finite_vals = vals[np.isfinite(vals)]
        if finite_vals.size > 0 and np.nanmin(finite_vals) >= 0 and np.nanmax(finite_vals) <= 172800:
            base = pd.Timestamp(fallback_base_date).normalize()
            return base + pd.to_timedelta(vals, unit="s")

    return pd.to_datetime(vals, errors="coerce")


if not space_weather_file or not isinstance(space_weather_file, str):
    raise ValueError("Set space_weather_file (or filename_noaa_geoE) to a valid NOAA GeoE NetCDF path")

# Open without auto time decoding so we can explicitly decode.
ds = xr.open_dataset(space_weather_file, decode_times=False)

# Infer fallback date from filename (YYYYMMDD), if present.
m = re.search(r"(\d{8})", space_weather_file)
fallback_base_date = pd.Timestamp(m.group(1)) if m else pd.Timestamp("2024-01-01")

time_name = "time" if "time" in ds.coords else ("time" if "time" in ds.data_vars else None)
if time_name is None:
    raise ValueError("No time coordinate/variable named 'time' found in file")

decoded_times = decode_time_index(ds, time_name, fallback_base_date)
finite_times = decoded_times[~pd.isna(decoded_times)]

print("File:", space_weather_file)
print("Time attrs:", dict(ds[time_name].attrs))
print("Decoded finite timestamps:", len(finite_times), "of", len(decoded_times))
if len(finite_times) > 0:
    print("Decoded min:", finite_times.min())
    print("Decoded max:", finite_times.max())
    print("First 20 decoded timestamps:")
    print(finite_times[:20])

    if len(finite_times) > 1:
        dt_seconds = np.diff(finite_times) / np.timedelta64(1, "s")
        dt_seconds = np.asarray(dt_seconds, dtype=float)
        dt_seconds = dt_seconds[np.isfinite(dt_seconds)]
        if dt_seconds.size > 0:
            print("Unique timestep seconds (first 10):", np.unique(dt_seconds)[:10])

ds.close()




## GeoE Movie for May 11, 2024
Load NOAA GeoElectric files for May 11, 2024 and render an animation with Ex on the left and Ey on the right, including U.S. state boundaries.

In [ ]:
from pathlib import Path
import os
import re
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import geopandas as gpd
from matplotlib import animation

START_DATE = pd.Timestamp("2024-05-11")
END_DATE = pd.Timestamp("2024-05-11")

# Candidate roots for local NOAA GeoE NetCDF files.
candidate_dirs = [
    Path("data/space_weather/NOAA_geoE"),
    Path("data/NOAA_geoE"),
    Path("../data/space_weather/NOAA_geoE"),
    Path("../data/NOAA_geoE"),
    Path("/Users/ryanmc/Documents/NASA_JPL/Projects/NaturalHazards/NASA ROSES Disasters 2025-2027/data/space_weather/NOAA_geoE"),
]

# Candidate sources for U.S. state boundaries.
state_boundary_candidates = [
    Path("/Users/ryanmc/Documents/Conferences/Jack_Eddy_Symposium_2022/dev/location_data/Census_Bureau_Data/tl_2014_us_state/tl_2014_us_state.shp"),
    Path("data/CA_location_data/CA_State.shp"),
    Path("../data/CA_location_data/CA_State.shp"),
    Path("data/selected_states_May2024_event.geojson"),
    Path("data/selected_states_MarchApril2023_event.geojson"),
    Path("data/ca_and_bordering_states.geojson"),
    Path("../data/selected_states_May2024_event.geojson"),
    Path("../data/selected_states_MarchApril2023_event.geojson"),
    Path("../data/ca_and_bordering_states.geojson"),
]

# CONUS geographic bounds (approximate).
CONUS_LON_MIN, CONUS_LON_MAX = -125.0, -66.5
CONUS_LAT_MIN, CONUS_LAT_MAX = 24.0, 50.0

def pick_name(candidates, options):
    for cand in candidates:
        for name in options:
            if cand in name.lower():
                return name
    return None

def find_daily_files(start_date, end_date, roots):
    found = []
    for day in pd.date_range(start_date, end_date, freq="D"):
        ymd = day.strftime("%Y%m%d")
        day_matches = []
        for root in roots:
            if root.exists():
                day_matches.extend(sorted(root.glob(f"{ymd}*.nc")))
        if len(day_matches) == 0:
            raise FileNotFoundError(f"No NOAA GeoE file found for {ymd}. Checked roots: {[str(r) for r in roots]}")
        found.append(day_matches[0])
    return found

def infer_base_date_from_files(files, default_date):
    for fp in files:
        m = re.search(r"(\d{8})", fp.name)
        if m:
            try:
                return pd.Timestamp(m.group(1))
            except Exception:
                continue
    return pd.Timestamp(default_date).normalize()

def load_state_boundaries(candidates):
    for shp in candidates:
        if shp.exists():
            gdf = gpd.read_file(shp)
            if gdf.crs is not None and str(gdf.crs) != "EPSG:4326":
                gdf = gdf.to_crs("EPSG:4326")
            print(f"Using state boundaries: {shp}")
            return gdf
    print("No state boundary file found; continuing without boundaries.")
    return None

def filter_to_conus_states(gdf):
    if gdf is None or gdf.empty:
        return gdf

    out = gdf.copy()
    drop_codes = {"AK", "HI"}
    code_cols = ["STUSPS", "STUSPS10", "STATE_ABBR", "ABBREV", "STUSAB"]
    name_cols = ["NAME", "STATE_NAME", "NAMELSAD", "STATE"]

    dropped = False
    for col in code_cols:
        if col in out.columns:
            out = out[~out[col].astype(str).str.upper().isin(drop_codes)]
            dropped = True
            break

    if not dropped:
        for col in name_cols:
            if col in out.columns:
                out = out[~out[col].astype(str).str.lower().isin({"alaska", "hawaii"})]
                dropped = True
                break

    return out

def decode_time_index(ds, time_name, fallback_base_date):
    time_da = ds[time_name]

    if np.issubdtype(time_da.dtype, np.datetime64):
        return pd.to_datetime(time_da.values)

    # Try CF decoding (uses units/calendar attrs when present).
    try:
        decoded = xr.decode_cf(ds[[time_name]])
        vals = decoded[time_name].values
        if np.issubdtype(vals.dtype, np.datetime64):
            return pd.to_datetime(vals)
    except Exception:
        pass

    vals = np.asarray(time_da.values)

    # Numeric fallback: use declared units first, then safe heuristics.
    if np.issubdtype(vals.dtype, np.number):
        units_attr = str(time_da.attrs.get("units", "")).lower()

        if "since" in units_attr:
            unit_map = {
                "seconds": "s", "second": "s", "sec": "s",
                "minutes": "m", "minute": "m", "min": "m",
                "hours": "h", "hour": "h", "hr": "h",
                "days": "D", "day": "D",
                "milliseconds": "ms", "millisecond": "ms", "msec": "ms",
                "microseconds": "us", "microsecond": "us", "usec": "us",
                "nanoseconds": "ns", "nanosecond": "ns", "nsec": "ns",
            }
            parsed_unit = None
            for key, val in unit_map.items():
                if key in units_attr:
                    parsed_unit = val
                    break
            if parsed_unit is not None:
                try:
                    base = units_attr.split("since", 1)[1].strip()
                    origin = pd.Timestamp(base)
                    return pd.to_datetime(vals, unit=parsed_unit, origin=origin)
                except Exception:
                    pass

        # Heuristic when units metadata is missing or malformed.
        for unit in ("s", "ms", "us", "ns", "h", "m"):
            try:
                candidate = pd.to_datetime(vals, unit=unit, origin="unix")
                if candidate.min() >= pd.Timestamp("2000-01-01") and candidate.max() <= pd.Timestamp("2100-01-01"):
                    return candidate
            except Exception:
                continue

        # NOAA GeoE files sometimes store seconds-of-day without date metadata.
        if np.nanmin(vals) >= 0 and np.nanmax(vals) <= 172800:
            base = pd.Timestamp(fallback_base_date).normalize()
            return base + pd.to_timedelta(vals, unit="s")

    # Last resort for string/object arrays.
    return pd.to_datetime(vals, errors="coerce")

def infer_time_and_point_dims(da, time_name):
    time_dim = time_name if time_name in da.dims else None
    point_dim = None
    for dim in da.dims:
        if dim == time_dim:
            continue
        if da.sizes[dim] > 1:
            point_dim = dim
            break
    if point_dim is None:
        raise ValueError(f"Could not infer point dimension for {da.name}; dims={da.dims}, sizes={dict(da.sizes)}")
    return time_dim, point_dim

def to_point_vector(da, point_dim, time_name):
    work = da
    if time_name in work.dims:
        work = work.isel({time_name: 0})
    for dim in list(work.dims):
        if dim != point_dim:
            work = work.isel({dim: 0})
    work = work.squeeze(drop=True)
    if work.ndim != 1:
        raise ValueError(f"Expected 1D point vector, got {work.ndim}D for {da.name} with shape={work.shape}")
    return np.asarray(work.values, dtype=float)

def to_time_point_matrix(da, time_name, point_dim):
    work = da
    for dim in list(work.dims):
        if dim not in (time_name, point_dim):
            work = work.isel({dim: 0})
    work = work.transpose(time_name, point_dim)
    vals = np.asarray(work.values, dtype=float)
    if vals.ndim != 2:
        raise ValueError(f"Expected 2D time-point array, got shape={vals.shape} for {da.name}")
    return vals

geoe_files = find_daily_files(START_DATE, END_DATE, candidate_dirs)
print("Using NOAA GeoE files:")
for f in geoe_files:
    print(f"  - {f}")
base_date_for_time = infer_base_date_from_files(geoe_files, START_DATE)

state_boundaries_gdf = load_state_boundaries(state_boundary_candidates)
state_boundaries_conus_gdf = filter_to_conus_states(state_boundaries_gdf)

datasets = [xr.open_dataset(fp) for fp in geoe_files]
try:
    ds_movie = xr.concat(datasets, dim="time", combine_attrs="drop_conflicts")
finally:
    for d in datasets:
        d.close()

# Ensure time is sorted and unique after concatenation.
ds_movie = ds_movie.sortby("time")
time_vals = decode_time_index(ds_movie, "time", fallback_base_date=base_date_for_time)
_, unique_idx = np.unique(np.asarray(time_vals), return_index=True)
if len(unique_idx) < len(time_vals):
    ds_movie = ds_movie.isel(time=np.sort(unique_idx))
    time_vals = decode_time_index(ds_movie, "time", fallback_base_date=base_date_for_time)

coord_names = list(ds_movie.coords)
var_names = list(ds_movie.data_vars)
time_name = pick_name(["time"], coord_names)
lat_name = pick_name(["latitude", "lat"], var_names) or pick_name(["latitude", "lat"], coord_names)
lon_name = pick_name(["longitude", "lon", "long"], var_names) or pick_name(["longitude", "lon", "long"], coord_names)
ex_name = pick_name(["ex", "e_x"], var_names)
ey_name = pick_name(["ey", "e_y"], var_names)

if None in (time_name, lat_name, lon_name, ex_name, ey_name):
    raise ValueError(
        f"Could not identify required fields. coords={coord_names}, vars={var_names}, "
        f"detected time={time_name}, lat={lat_name}, lon={lon_name}, Ex={ex_name}, Ey={ey_name}"
    )

_, point_dim_ex = infer_time_and_point_dims(ds_movie[ex_name], time_name)
_, point_dim_ey = infer_time_and_point_dims(ds_movie[ey_name], time_name)
if point_dim_ex != point_dim_ey:
    raise ValueError(f"Ex/Ey point dimension mismatch: {point_dim_ex} vs {point_dim_ey}")
point_dim = point_dim_ex

lat_vec = to_point_vector(ds_movie[lat_name], point_dim, time_name)
lon_vec = to_point_vector(ds_movie[lon_name], point_dim, time_name)
ex_mat = to_time_point_matrix(ds_movie[ex_name], time_name, point_dim)
ey_mat = to_time_point_matrix(ds_movie[ey_name], time_name, point_dim)

valid_points = np.isfinite(lat_vec) & np.isfinite(lon_vec)
valid_points = valid_points & np.isfinite(ex_mat).any(axis=0) & np.isfinite(ey_mat).any(axis=0)

# Keep only CONUS points for plotting/movie export.
conus_points = (
    (lon_vec >= CONUS_LON_MIN) & (lon_vec <= CONUS_LON_MAX) &
    (lat_vec >= CONUS_LAT_MIN) & (lat_vec <= CONUS_LAT_MAX)
)
valid_points = valid_points & conus_points

if not np.any(valid_points):
    raise ValueError("No valid CONUS points remain after filtering; check dataset coverage or bounds.")

lat_vec = lat_vec[valid_points]
lon_vec = lon_vec[valid_points]
ex_mat = ex_mat[:, valid_points]
ey_mat = ey_mat[:, valid_points]

lat_unique = np.unique(lat_vec)
lon_unique = np.unique(lon_vec)
lat_idx = np.searchsorted(lat_unique, lat_vec)
lon_idx = np.searchsorted(lon_unique, lon_vec)

nt = ex_mat.shape[0]
ny = len(lat_unique)
nx = len(lon_unique)

ex_cube = np.full((nt, ny, nx), np.nan, dtype=float)
ey_cube = np.full((nt, ny, nx), np.nan, dtype=float)
for t in range(nt):
    ex_cube[t, lat_idx, lon_idx] = ex_mat[t]
    ey_cube[t, lat_idx, lon_idx] = ey_mat[t]

all_vals = np.r_[ex_cube[np.isfinite(ex_cube)], ey_cube[np.isfinite(ey_cube)]]
cmax = float(np.nanpercentile(np.abs(all_vals), 99)) if all_vals.size > 0 else 1.0
if cmax == 0:
    cmax = 1.0

fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

im_ex = axes[0].imshow(
    ex_cube[0],
    origin="lower",
    extent=[float(lon_unique.min()), float(lon_unique.max()), float(lat_unique.min()), float(lat_unique.max())],
    cmap="RdBu_r",
    vmin=-cmax,
    vmax=cmax,
    interpolation="nearest",
    aspect="auto",
    zorder=1,
)
im_ey = axes[1].imshow(
    ey_cube[0],
    origin="lower",
    extent=[float(lon_unique.min()), float(lon_unique.max()), float(lat_unique.min()), float(lat_unique.max())],
    cmap="RdBu_r",
    vmin=-cmax,
    vmax=cmax,
    interpolation="nearest",
    aspect="auto",
    zorder=1,
)

if state_boundaries_conus_gdf is not None and not state_boundaries_conus_gdf.empty:
    for ax in axes:
        state_boundaries_conus_gdf.boundary.plot(
            ax=ax,
            color="black",
            linewidth=0.45,
            alpha=0.7,
            zorder=3,
        )

axes[0].set_title(f"{ex_name} [mV/km] (CONUS)")
axes[1].set_title(f"{ey_name} [mV/km] (CONUS)")
for ax in axes:
    ax.set_xlim(CONUS_LON_MIN, CONUS_LON_MAX)
    ax.set_ylim(CONUS_LAT_MIN, CONUS_LAT_MAX)
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")
    ax.grid(True, linestyle="--", alpha=0.2)

cbar = fig.colorbar(im_ey, ax=axes.ravel().tolist(), shrink=0.92)
cbar.set_label("GeoElectric field (mV/km)")
title = fig.suptitle("")

def update(frame):
    im_ex.set_data(ex_cube[frame])
    im_ey.set_data(ey_cube[frame])
    title.set_text(f"NOAA GeoE CONUS: {pd.Timestamp(time_vals[frame]).isoformat()}")
    return im_ex, im_ey, title

ani = animation.FuncAnimation(fig, update, frames=nt, interval=200, blit=False)

out_dir = Path("notebooks/outputs")
out_dir.mkdir(parents=True, exist_ok=True)
mp4_path = out_dir / "geoe_ex_ey_20240511_conus.mp4"
gif_path = out_dir / "geoe_ex_ey_20240511_conus.gif"

if animation.writers.is_available("ffmpeg"):
    writer = animation.FFMpegWriter(fps=6, bitrate=2400)
    ani.save(mp4_path, writer=writer, dpi=140)
    print(f"Saved movie: {mp4_path}")
else:
    ani.save(gif_path, writer="pillow", fps=6, dpi=140)
    print(f"ffmpeg not available; saved GIF instead: {gif_path}")

print(f"Frame time range: {time_vals.min()} to {time_vals.max()}")
plt.close(fig)